<a href="https://colab.research.google.com/github/arulperiyannagounder-collab/Training_Hands_on/blob/main/Resume_analysis_via_Chroma_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetry-p

In [3]:
!pip install pytesseract
!apt-get install tesseract-ocr -y
!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [4]:
from google.colab import files

uploaded = files.upload()

Saving Arul resume (1).pdf to Arul resume (1).pdf


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pypdf import PdfReader

documents = []

for file_name in uploaded.keys():
    pdf = PdfReader(file_name)

    text = ""

    for page in pdf.pages:
        text += page.extract_text()

    documents.append({
        "resume_name": file_name,
        "content": text
    })

In [6]:
print("Number of resumes:", len(documents))

for doc in documents:
    print(doc["resume_name"])

Number of resumes: 1
Arul resume (1).pdf


In [7]:
def chunk_text(text, chunk_size=500, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [8]:
chunks = []

for doc in documents:

    split_text = chunk_text(doc["content"])

    for chunk in split_text:

        chunks.append({
            "resume_name": doc["resume_name"],
            "text": chunk
        })

print("Total Chunks:", len(chunks))

Total Chunks: 8


In [9]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

In [ ]:
print(embeddings.shape)

In [ ]:
!pip install -q chromadb

In [22]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="resume_collection"
)

In [23]:
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(texts))],
    metadatas=[
        {"resume": chunk["resume_name"]}
        for chunk in chunks
    ]
)

In [24]:
collection.count()

8

In [18]:
query = "Which candidate is best suited for a Data Scientist role?"

In [19]:
query_embedding = embedding_model.encode(
    query
).tolist()

In [25]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

In [27]:
print(results["documents"][0])

[' / i n / a r u l - p - 9 a 5 3 6 1 3 3 1\n+ 9 1  8 5 0 8 1 9 2 2 4 4  \nA B O U T\n Motivat ed Engineering student wit h a str ong f ocus on Ar tificial Int elligence, Data science, Machine Learning,\nand Deep Learning. Pr o v en ability t o build functional AI solutions and RESTful APIs t o solv e comple x, r eal-\nw orld pr oblems.  \nK a r p a g a m  C o l l e g e  o f  E n g i n e e r i n g  , C o i m b a t o r e\n B . T e c h  A r t i f i c i a l  I n t e l l i g e n c e  &  D a t a  S c i e n c e', 'C o m p u t e r  V i s i o n .\n· T o o l s :      P o s t g r e s , D o c k e r , K a f k a , R o b o f l o w ,\nS n o w f l a k e\nE D U C A T I O N\nC E R T I F I C A T I O N S\nA W S  E d u c a t e  C l o u d  C o m p u t i n g\n  a r u l p e r i y a n n a g o u n d e r @ g m a i l . c o m    \ng i t h u b . c o m / a r u l p e r i y a n n a g o u n d e r - c o l l a b\nw w w . l i n k e d i n . c o m / i n / a r u l - p - 9 a 5 3 6 1 3 3 1\n+ 9 1  8 5 0 8 1 9 2 2 4 4  \nA B O U

In [26]:
from groq import Groq

client = Groq(
    api_key=""
)

retrieved_chunks = results["documents"][0]

context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

In [28]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Based on the provided context, the candidate "Arul Peryannan Goundar" is best suited for a Data Scientist role. This is evident from the following points:

1. The candidate has a strong focus on Data Science, Machine Learning, and Artificial Intelligence.
2. The candidate has experience in building functional AI solutions and RESTful APIs to solve complex real-world problems.
3. The candidate has a strong educational background in Data Science and Artificial Intelligence, with a CGPA of 8.2 from B.Tech Artificial Intelligence & Data Science at Karappa College of Engineering, Coimbatore.
4. The candidate has hands-on experience with various tools and technologies such as Postgres, Docker, Kafka, Roboflow, and Snowflake.
5. The candidate has worked on projects related to Data Science and Artificial Intelligence, such as building a Convolutional Neural Network (CNN) to classify vehicle images and designing an intelligent decision logic layer to evaluate prediction reliability.

Overall, t

In [29]:
prompt = f"""
You are an HR recruiter.

Based ONLY on the resume information provided below,
identify which candidate is best suited for a Generative AI Engineer role.

Explain:
1. Candidate name
2. Relevant skills
3. Relevant projects
4. Certifications
5. Why the candidate is the best fit

Context:
{context}

Question:
{query}
"""

In [30]:
for doc in documents:
    print(doc["resume_name"])
    print(len(doc["content"]))
    print("-"*50)

Arul resume (1).pdf
3095
--------------------------------------------------
